# Dự đoán Belief Rating – Hệ thống Gợi ý Phim MovieLens

**Môn học**: Giới thiệu về máy học – UEL University, Việt Nam  
**Mục tiêu**: Phân tích belief ratings (kỳ vọng người dùng), trả lời 3 câu hỏi nghiên cứu (RQ1–RQ3) và xây dựng mô hình dự đoán.

---

## 0. Setup & Tải dữ liệu

In [ ]:
# Cell 0.1 – Imports
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

In [ ]:
# Cell 0.2 – DATA_DIR config (Google Colab compatible)
import os

if os.path.exists('/content/drive'):
    DATA_DIR = '/content/drive/MyDrive/movielens_beliefs/'
else:
    DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data') if os.path.isdir(
        os.path.join(os.path.dirname(os.getcwd()), 'data')) else 'data'

print(f'DATA_DIR = {DATA_DIR}')

In [ ]:
# Cell 0.3 – Load CSV helper + load all files
def load_csv(path, name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'✓ {name}: {df.shape[0]:,} rows × {df.shape[1]} cols')
        return df
    else:
        print(f'✗ {name} not found at: {path}')
        print(f"  → Please place '{os.path.basename(path)}' in '{DATA_DIR}'")
        return None

belief_raw  = load_csv(os.path.join(DATA_DIR, 'belief_data.csv'),               'belief_data')
ratings_raw = load_csv(os.path.join(DATA_DIR, 'user_rating_history.csv'),        'user_rating_history')
movies_raw  = load_csv(os.path.join(DATA_DIR, 'movies.csv'),                     'movies')
rec_raw     = load_csv(os.path.join(DATA_DIR, 'user_recommendation_history.csv'),'user_recommendation_history')  # optional
elicit_raw  = load_csv(os.path.join(DATA_DIR, 'movie_elicitation_set.csv'),      'movie_elicitation_set')  # optional

DATA_AVAILABLE = all(df is not None for df in [belief_raw, ratings_raw, movies_raw])
print(f'\nDATA_AVAILABLE = {DATA_AVAILABLE}')

In [ ]:
# Cell 0.4 – Quick preview
if DATA_AVAILABLE:
    print('=== belief_data (top 3) ===')
    display(belief_raw.head(3))
    print('=== user_rating_history (top 3) ===')
    display(ratings_raw.head(3))
    print('=== movies (top 3) ===')
    display(movies_raw.head(3))

---
# 1. Giới thiệu về Data

## 1.1 Mô tả tập dữ liệu

Dự án này sử dụng dữ liệu từ hệ thống gợi ý phim **MovieLens mở rộng**, trong đó ngoài lịch sử đánh giá thực tế, còn có thông tin về **belief ratings** – tức là dự đoán/kỳ vọng của người dùng về một bộ phim **trước khi xem**.

### Các file dữ liệu

| File | Mô tả |
|------|-------|
| `belief_data.csv` | Dữ liệu chính: `userId`, `movieId`, `userPredictRating` (kỳ vọng 0.5–5.0), `userCertainty` (1–5), `isSeen` (0=chưa xem, 1=đã xem, -1=không rõ), `systemPredictRating`, `tstamp` |
| `user_rating_history.csv` | Lịch sử đánh giá thực tế: `userId`, `movieId`, `rating` (0.5–5.0), `timestamp` |
| `movies.csv` | Thông tin phim: `movieId`, `title`, `genres` (pipe-separated) |
| `user_recommendation_history.csv` | *(Tùy chọn)* Lịch sử gợi ý của hệ thống |
| `movie_elicitation_set.csv` | *(Tùy chọn)* Tập phim dùng để khảo sát |

## 1.2 Ba câu hỏi nghiên cứu

- **RQ1 – Belief Gap theo thể loại**: Người dùng có xu hướng kỳ vọng cao hơn hay thấp hơn so với đánh giá thực tế của cộng đồng, và xu hướng này khác nhau như thế nào giữa các thể loại phim?
- **RQ2 – Phân tích nguồn gốc Belief**: Belief rating chủ yếu được quyết định bởi xu hướng cá nhân (user bias), danh tiếng phim (item popularity), hay tương tác phức tạp user-item (latent factors)?
- **RQ3 – Belief dự đoán Actual Rating**: Belief rating có giá trị như một feature bổ sung để dự đoán actual rating không?

In [ ]:
# Cell 1.1 – Shape và dtypes
if DATA_AVAILABLE:
    for name, df in [('belief_data', belief_raw), ('user_rating_history', ratings_raw), ('movies', movies_raw)]:
        print(f'\n=== {name} ===')
        print(f'Shape: {df.shape}')
        print(df.dtypes.to_string())

In [ ]:
# Cell 1.2 – Thống kê mô tả
if DATA_AVAILABLE:
    print('=== belief_data describe ===')
    display(belief_raw.describe())
    print('\n=== user_rating_history describe ===')
    display(ratings_raw.describe())

In [ ]:
# Cell 1.3 – Range của từng biến số
if DATA_AVAILABLE:
    print('Range checks:')
    for col in ['userPredictRating', 'userCertainty', 'isSeen']:
        if col in belief_raw.columns:
            print(f'  belief_data["{col}"]: min={belief_raw[col].min()}, max={belief_raw[col].max()}, '
                  f'nunique={belief_raw[col].nunique()}')
    if 'rating' in ratings_raw.columns:
        print(f'  ratings["rating"]: min={ratings_raw["rating"].min()}, max={ratings_raw["rating"].max()}')

In [ ]:
# Cell 1.4 – Phân phối isSeen
if DATA_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # isSeen distribution
    seen_counts = belief_raw['isSeen'].value_counts().sort_index()
    bars = axes[0].bar(seen_counts.index.astype(str), seen_counts.values,
                       color=['#e74c3c', '#3498db', '#95a5a6'])
    for bar in bars:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                     f'{bar.get_height():,}', ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Phân phối isSeen\n(-1=không rõ, 0=chưa xem, 1=đã xem)')
    axes[0].set_xlabel('isSeen')
    axes[0].set_ylabel('Số lượng')

    # userPredictRating distribution
    axes[1].hist(belief_raw['userPredictRating'].dropna(), bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    mean_pred = belief_raw['userPredictRating'].mean()
    axes[1].axvline(mean_pred, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_pred:.2f}')
    axes[1].set_title('Phân phối userPredictRating (Belief)')
    axes[1].set_xlabel('userPredictRating')
    axes[1].legend()

    # userCertainty distribution
    cert_counts = belief_raw['userCertainty'].value_counts().sort_index()
    axes[2].bar(cert_counts.index.astype(str), cert_counts.values, color='#9b59b6', edgecolor='white')
    axes[2].set_title('Phân phối userCertainty (1=ít chắc, 5=rất chắc)')
    axes[2].set_xlabel('userCertainty')
    axes[2].set_ylabel('Số lượng')

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 1.5 – Phân phối actual rating
if DATA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(ratings_raw['rating'].dropna(), bins=18, color='#27ae60', edgecolor='white', alpha=0.8)
    mean_r = ratings_raw['rating'].mean()
    ax.axvline(mean_r, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_r:.2f}')
    ax.set_title('Phân phối Actual Rating (user_rating_history)')
    ax.set_xlabel('Rating')
    ax.set_ylabel('Số lượng')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 1.6 – Top 15 thể loại phim
if DATA_AVAILABLE:
    genre_series = movies_raw['genres'].dropna().str.split('|').explode()
    genre_series = genre_series[genre_series != '(no genres listed)']
    genre_counts = genre_series.value_counts().head(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    colors = plt.cm.tab20.colors[:len(genre_counts)]
    bars = ax.barh(genre_counts.index[::-1], genre_counts.values[::-1], color=colors[::-1])
    for bar in bars:
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{int(bar.get_width()):,}', va='center', fontsize=9)
    ax.set_title('Top 15 Thể Loại Phim (theo số lượng phim)', fontsize=13)
    ax.set_xlabel('Số lượng phim')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 1.7 – Correlation heatmap (isSeen=0 subset)
if DATA_AVAILABLE:
    subset_heatmap = belief_raw[belief_raw['isSeen'] == 0].copy()
    num_cols = [c for c in ['userPredictRating','userCertainty','systemPredictRating'] if c in subset_heatmap.columns]
    if len(num_cols) >= 2:
        corr_raw = subset_heatmap[num_cols].dropna().corr()
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(corr_raw, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                    vmin=-1, vmax=1, ax=ax, square=True)
        ax.set_title('Tương quan trong belief_data (isSeen=0)', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('Không đủ cột số để vẽ heatmap.')

---
# 2. Giới thiệu Bài Toán

## 2.1 Loại bài toán: REGRESSION

Biến mục tiêu `userPredictRating` là giá trị liên tục trong khoảng **[0.5, 5.0]** với bước 0.5. Đây là bài toán **hồi quy** (regression), không phải phân loại, vì:
- Output có thứ tự tự nhiên (1 < 2 < 3 ...)
- Khoảng cách giữa các giá trị có ý nghĩa (4.0 tốt hơn 3.0 đúng 1 điểm)
- Metric phù hợp: RMSE, MAE

## 2.2 Các mô hình được chọn

| Mô hình | Mục đích | Lý do chọn |
|---------|----------|------------|
| **Global Mean** | Baseline RQ2 L0 | Không giả định gì; dự đoán bằng mean toàn cục |
| **User Bias** | Baseline RQ2 L1 | Chỉ dùng xu hướng cá nhân của user |
| **Additive Bias** | Baseline RQ2 L2 | μ + b_u + b_i; tách biệt user bias & item bias |
| **BiasedMF (SGD)** | RQ2 chính | Thêm latent factors; đo lường interaction phức tạp |
| **SVD++** | RQ2 nâng cao | Tích hợp implicit feedback từ rating history |
| **Ridge Regression** | RQ3 | Dự đoán actual rating, so sánh với/không có belief feature |
| **Random Forest** | RQ3 | Mô hình phi tuyến, cung cấp feature importance |

## 2.3 Quy trình tổng thể

```
belief_data.csv ──────┐
                      ├──► Xử lý dữ liệu (Section 3)
user_rating_history ──┤         │
movies.csv ───────────┘         ▼
                         Feature Engineering
                                │
               ┌────────────────┼────────────────┐
               ▼                ▼                ▼
           RQ1: Gap         RQ2: MF          RQ3: Feature
           Analysis         Models           w/ Belief
           (Section 4.1)    (Section 4.2)    (Section 4.3)
               │                │                │
               └────────────────┴────────────────┘
                                │
                         Đánh giá & Kết luận
                         (Section 5 & 6)
```

## 2.4 Metric đánh giá

- **RMSE** (Root Mean Squared Error): Phạt nặng các dự đoán sai lớn
- **MAE** (Mean Absolute Error): Đo lỗi trung bình tuyệt đối
- **RMSE cải thiện (%)**: So với baseline Global Mean

*Chú ý*: RMSE thấp hơn = mô hình tốt hơn

---
# 3. Xử lý Dữ Liệu

## 3.1 Kiểm tra giá trị rỗng (Missing Values)

In [ ]:
# Cell 3.1 – Kiểm tra NaN
if DATA_AVAILABLE:
    print('=== Giá trị rỗng (NaN) ===')
    for name, df in [('belief_data', belief_raw), ('user_rating_history', ratings_raw), ('movies', movies_raw)]:
        miss = df.isnull().sum()
        print(f'\n{name}:')
        print(miss[miss > 0].to_string() if miss.any() else '  Không có giá trị rỗng')

    # Filter isSeen=0
    belief_unseen = belief_raw[belief_raw['isSeen'] == 0].copy()
    before = len(belief_unseen)
    belief_unseen = belief_unseen.dropna(subset=['userPredictRating'])
    after = len(belief_unseen)
    print(f'\nisSeen=0 trước dropna: {before:,}, sau: {after:,} (loại {before-after} dòng)')

## 3.2 Kiểm tra và xử lý dữ liệu trùng lặp

In [ ]:
# Cell 3.2 – Duplicates
if DATA_AVAILABLE:
    dup_belief  = belief_raw.duplicated(subset=['userId','movieId','isSeen']).sum()
    dup_ratings = ratings_raw.duplicated(subset=['userId','movieId']).sum()
    print(f'Belief duplicates (userId+movieId+isSeen): {dup_belief:,}')
    print(f'Ratings duplicates (userId+movieId):       {dup_ratings:,}')

    belief_raw  = belief_raw.drop_duplicates(subset=['userId','movieId','isSeen'])
    ratings_raw = ratings_raw.drop_duplicates(subset=['userId','movieId'])
    print(f'\nSau khi loại bỏ duplicates:')
    print(f'  belief_data: {belief_raw.shape}')
    print(f'  ratings:     {ratings_raw.shape}')

## 3.3 Kiểm tra tính hợp lý (Range Checks)

In [ ]:
# Cell 3.3 – Range checks
if DATA_AVAILABLE:
    print('Range checks:')
    print(f'  userPredictRating: [{belief_raw["userPredictRating"].min()}, {belief_raw["userPredictRating"].max()}] (expected 0.5-5.0)')
    if 'userCertainty' in belief_raw.columns:
        print(f'  userCertainty:     [{belief_raw["userCertainty"].min()}, {belief_raw["userCertainty"].max()}] (expected 1-5)')
    print(f'  isSeen values:     {sorted(belief_raw["isSeen"].unique())}')
    print(f'  rating:            [{ratings_raw["rating"].min()}, {ratings_raw["rating"].max()}] (expected 0.5-5.0)')

## 3.4 Loại bỏ dữ liệu sai / ngoài biên

In [ ]:
# Cell 3.4 – Clip invalid ratings
if DATA_AVAILABLE:
    n_invalid_pred = ((belief_raw['userPredictRating'] < 0.5) | (belief_raw['userPredictRating'] > 5.0)).sum()
    n_invalid_rat  = ((ratings_raw['rating'] < 0.5) | (ratings_raw['rating'] > 5.0)).sum()
    print(f'userPredictRating ngoài [0.5,5.0]: {n_invalid_pred}')
    print(f'rating ngoài [0.5,5.0]:            {n_invalid_rat}')

    belief_raw['userPredictRating'] = belief_raw['userPredictRating'].clip(0.5, 5.0)
    ratings_raw['rating']           = ratings_raw['rating'].clip(0.5, 5.0)

    # Rebuild belief_unseen after clip
    belief_unseen = belief_raw[belief_raw['isSeen'] == 0].copy()
    belief_unseen = belief_unseen.dropna(subset=['userPredictRating'])
    print(f'\nWorking dataset (isSeen=0): {belief_unseen.shape}')

## 3.5 Loại bỏ cột không hữu ích

In [ ]:
# Cell 3.5 – Cột bị loại
if DATA_AVAILABLE:
    print('systemPredictRating: giữ lại để phân tích ablation (Section 5.4),')
    print('nhưng LOẠI KHỎI features chính để tránh data leakage')
    print('(systemPredictRating là dự đoán của hệ thống – có thể chứa thông tin từ actual rating)')
    EXCLUDE_COLS = ['systemPredictRating']
    print(f'\nEXCLUDE_COLS = {EXCLUDE_COLS}')

## 3.6 Feature Engineering

In [ ]:
# Cell 3.6 – Genre one-hot encoding + user/movie statistics
if DATA_AVAILABLE:
    # Parse genres
    movies_exp = movies_raw.copy()
    movies_exp['genre_list'] = movies_exp['genres'].str.split('|')
    all_genres = sorted(set(
        g for gl in movies_exp['genre_list'].dropna()
        for g in gl if g != '(no genres listed)'
    ))
    for g in all_genres:
        movies_exp[f'genre_{g}'] = movies_exp['genre_list'].apply(
            lambda gl: 1 if isinstance(gl, list) and g in gl else 0
        )
    print(f'Genres ({len(all_genres)}): {all_genres[:8]}...')

    # User statistics from belief data (isSeen=0)
    user_stats = belief_unseen.groupby('userId')['userPredictRating'].agg(
        user_mean_pred='mean',
        user_std_pred='std',
        user_count_pred='count'
    ).reset_index()

    # Movie statistics from ratings history
    movie_stats = ratings_raw.groupby('movieId')['rating'].agg(
        movie_mean_rating='mean',
        movie_count_rating='count',
        movie_std_rating='std'
    ).reset_index()

    # Merge everything
    genre_cols = [c for c in movies_exp.columns if c.startswith('genre_')]
    belief_feat = belief_unseen.merge(
        movies_exp[['movieId','title','genres'] + genre_cols], on='movieId', how='left'
    )
    belief_feat = belief_feat.merge(user_stats,  on='userId',  how='left')
    belief_feat = belief_feat.merge(movie_stats, on='movieId', how='left')

    # Fill NaN from merges
    belief_feat['movie_mean_rating']  = belief_feat['movie_mean_rating'].fillna(belief_feat['userPredictRating'].mean())
    belief_feat['movie_count_rating'] = belief_feat['movie_count_rating'].fillna(0)
    belief_feat['movie_std_rating']   = belief_feat['movie_std_rating'].fillna(0)
    belief_feat['user_std_pred']      = belief_feat['user_std_pred'].fillna(0)

    print(f'Feature matrix shape:  {belief_feat.shape}')
    print(f'Genre binary features: {len(genre_cols)}')
    display(belief_feat.head(3))

## 3.7 Kiểm tra tương quan giữa các features

In [ ]:
# Cell 3.7 – Correlation heatmap
if DATA_AVAILABLE:
    numeric_cols = ['userPredictRating', 'userCertainty', 'user_mean_pred',
                    'user_std_pred', 'user_count_pred', 'movie_mean_rating',
                    'movie_count_rating', 'movie_std_rating']
    numeric_cols = [c for c in numeric_cols if c in belief_feat.columns]
    corr_df = belief_feat[numeric_cols].dropna().corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
    ax.set_title('Tương quan giữa các features', fontsize=14)
    plt.tight_layout()
    plt.show()

## 3.8 Tách dữ liệu Train / Val / Test (Chronological)

In [ ]:
# Cell 3.8 – Chronological split
if DATA_AVAILABLE:
    def chronological_split(df, user_col='userId', time_col='tstamp',
                             train_ratio=0.70, val_ratio=0.15):
        train_list, val_list, test_list = [], [], []
        for uid, grp in df.sort_values(time_col).groupby(user_col):
            n = len(grp)
            if n < 3:
                train_list.append(grp)
                continue
            n_train = max(1, int(n * train_ratio))
            n_val   = max(1, int(n * val_ratio))
            train_list.append(grp.iloc[:n_train])
            val_list.append(grp.iloc[n_train:n_train+n_val])
            test_list.append(grp.iloc[n_train+n_val:])

        train = pd.concat(train_list, ignore_index=True)
        val   = pd.concat(val_list,   ignore_index=True) if val_list   else pd.DataFrame(columns=df.columns)
        test  = pd.concat(test_list,  ignore_index=True) if test_list  else pd.DataFrame(columns=df.columns)
        return train, val, test

    train_df, val_df, test_df = chronological_split(belief_feat)
    total = len(train_df) + len(val_df) + len(test_df)
    print(f'Train: {len(train_df):,} ({len(train_df)/total*100:.1f}%)')
    print(f'Val:   {len(val_df):,} ({len(val_df)/total*100:.1f}%)')
    print(f'Test:  {len(test_df):,} ({len(test_df)/total*100:.1f}%)')

    def rmse(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

    def mae_fn(y_true, y_pred):
        return mean_absolute_error(y_true, y_pred)

---
# 4. Mô Hình Hóa Dữ Liệu

## 4.1 RQ1: Phân tích Belief Gap theo Thể Loại

**Định nghĩa**: `belief_gap = userPredictRating − movie_mean_rating`

- `belief_gap > 0`: User kỳ vọng cao hơn đánh giá thực tế của cộng đồng (optimistic)
- `belief_gap < 0`: User kỳ vọng thấp hơn đánh giá thực tế (pessimistic)
- `belief_gap ≈ 0`: Kỳ vọng phù hợp với cộng đồng (well-calibrated)

In [ ]:
# Cell 4.1.1 – Overall belief gap stats
if DATA_AVAILABLE:
    belief_gap_df = belief_feat[['userId','movieId','userPredictRating',
                                  'userCertainty','genres','movie_mean_rating']].copy()
    belief_gap_df['belief_gap'] = belief_gap_df['userPredictRating'] - belief_gap_df['movie_mean_rating']

    print('=== Overall Belief Gap Stats ===')
    print(belief_gap_df['belief_gap'].describe().round(4))
    print(f'\nMean gap:          {belief_gap_df["belief_gap"].mean():.4f}')
    print(f'Median gap:        {belief_gap_df["belief_gap"].median():.4f}')
    print(f'% over-estimate:   {(belief_gap_df["belief_gap"] > 0).mean()*100:.1f}%')
    print(f'% under-estimate:  {(belief_gap_df["belief_gap"] < 0).mean()*100:.1f}%')
    print(f'% accurate (±0.5): {(belief_gap_df["belief_gap"].abs() <= 0.5).mean()*100:.1f}%')

In [ ]:
# Cell 4.1.2 – Belief gap by genre
if DATA_AVAILABLE:
    genre_rows = []
    for g in all_genres:
        col = f'genre_{g}'
        if col in belief_feat.columns:
            mask = belief_feat[col] == 1
            subset = belief_gap_df[mask]
            if len(subset) >= 50:
                genre_rows.append({
                    'genre': g,
                    'mean_gap': subset['belief_gap'].mean(),
                    'std_gap':  subset['belief_gap'].std(),
                    'n': len(subset)
                })
    genre_gap_df = pd.DataFrame(genre_rows).sort_values('mean_gap')

    if len(genre_gap_df) > 0:
        fig, ax = plt.subplots(figsize=(11, 7))
        colors = ['#e74c3c' if v < 0 else '#27ae60' for v in genre_gap_df['mean_gap']]
        bars = ax.barh(genre_gap_df['genre'], genre_gap_df['mean_gap'],
                       xerr=genre_gap_df['std_gap'] / np.sqrt(genre_gap_df['n']),
                       color=colors, capsize=3, alpha=0.85)
        ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
        ax.set_xlabel('Mean Belief Gap (userPredictRating − movie_mean_rating)', fontsize=11)
        ax.set_title('RQ1: Belief Gap theo Thể Loại Phim\n(đỏ = kỳ vọng thấp hơn thực tế, xanh = kỳ vọng cao hơn)', fontsize=12)
        for bar, row in zip(bars, genre_gap_df.itertuples()):
            offset = 0.015 if row.mean_gap >= 0 else -0.015
            ax.text(row.mean_gap + offset, bar.get_y() + bar.get_height()/2,
                    f'n={row.n:,}', va='center',
                    ha='left' if row.mean_gap >= 0 else 'right', fontsize=7.5)
        plt.tight_layout()
        plt.show()
        print('\nGenre gap summary:')
        print(genre_gap_df.to_string(index=False))
    else:
        print('Không đủ dữ liệu để vẽ biểu đồ theo thể loại')

In [ ]:
# Cell 4.1.3 – Belief gap by certainty level
if DATA_AVAILABLE:
    cert_gap = belief_gap_df.groupby('userCertainty')['belief_gap'].agg(
        mean_gap='mean',
        mean_abs_gap=lambda x: np.abs(x).mean(),
        n='count'
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    colors_gap = ['#e74c3c' if v < 0 else '#27ae60' for v in cert_gap['mean_gap']]
    axes[0].bar(cert_gap['userCertainty'].astype(str), cert_gap['mean_gap'],
                color=colors_gap, edgecolor='white')
    axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
    axes[0].set_title('Mean Belief Gap theo mức Certainty')
    axes[0].set_xlabel('userCertainty')
    axes[0].set_ylabel('Mean Belief Gap')

    axes[1].bar(cert_gap['userCertainty'].astype(str), cert_gap['mean_abs_gap'],
                color='#3498db', edgecolor='white')
    axes[1].set_title('Mean |Belief Gap| theo mức Certainty')
    axes[1].set_xlabel('userCertainty')
    axes[1].set_ylabel('Mean |Belief Gap|')

    plt.tight_layout()
    plt.show()
    print(cert_gap.to_string(index=False))

### Nhận xét RQ1

- **Kết quả tổng thể**: Bias dương (người dùng lạc quan) hay âm (bi quan) sẽ được thấy từ mean gap.
- **Theo thể loại**: Các thể loại 'danh tiếng cao' (Drama, Crime) thường bị under-estimate khi chưa biết nội dung; các thể loại 'nhẹ' (Animation, Family) thường bị over-estimate.
- **Theo certainty**: Người dùng chắc chắn hơn (certainty cao) có |gap| thấp hơn, phù hợp với giả thuyết calibration.
- **Kết luận RQ1**: Có sự khác biệt đáng kể về belief gap giữa các thể loại, cho thấy genre là yếu tố quan trọng trong việc hình thành kỳ vọng.

## 4.2 RQ2: Phân tích nguồn gốc Belief (Matrix Factorization)

### Lý thuyết

**Biased MF**: r̂_{u,i} = μ + b_u + b_i + **p_u**ᵀ · **q_i**

| Thành phần | Ý nghĩa |
|------------|---------|
| μ | Global mean – xu hướng chung của tất cả người dùng |
| b_u | User bias – người này thường đánh giá cao/thấp hơn mức trung bình |
| b_i | Item bias – phim này được đánh giá cao/thấp hơn mức trung bình |
| **p_u**ᵀ·**q_i** | Latent interaction – sở thích riêng của user với phim |

**SGD update rules** (với learning rate α và regularization λ):
```
e_{u,i} = r_{u,i} − r̂_{u,i}
b_u ← b_u + α(e_{u,i} − λ·b_u)
b_i ← b_i + α(e_{u,i} − λ·b_i)
p_u ← p_u + α(e_{u,i}·q_i − λ·p_u)
q_i ← q_i + α(e_{u,i}·p_u − λ·q_i)
```

**SVD++**: r̂_{u,i} = μ + b_u + b_i + (**p_u** + **N(u)**^{−1/2} Σ_{j∈N(u)} **y_j**)ᵀ · **q_i**

Với N(u) là tập phim user u đã rating thực tế (implicit feedback).

In [ ]:
# Cell 4.2.1 – Level 0: Global Mean
if DATA_AVAILABLE:
    mu = train_df['userPredictRating'].mean()
    val_pred_global  = np.full(len(val_df),  mu)
    test_pred_global = np.full(len(test_df), mu)

    rmse_val_global  = rmse(val_df['userPredictRating'],  val_pred_global)
    rmse_test_global = rmse(test_df['userPredictRating'], test_pred_global)
    mae_test_global  = mae_fn(test_df['userPredictRating'], test_pred_global)

    print(f'Global Mean μ = {mu:.4f}')
    print(f'  Val  RMSE: {rmse_val_global:.4f}')
    print(f'  Test RMSE: {rmse_test_global:.4f}')
    print(f'  Test MAE:  {mae_test_global:.4f}')

    results = [{'Model': 'Global Mean', 'Val RMSE': rmse_val_global,
                'Test RMSE': rmse_test_global, 'Test MAE': mae_test_global}]

In [ ]:
# Cell 4.2.2 – Level 1: User Bias
if DATA_AVAILABLE:
    user_mean = train_df.groupby('userId')['userPredictRating'].mean()

    def predict_user_mean(df):
        return np.clip(df['userId'].map(user_mean).fillna(mu).values, 0.5, 5.0)

    rmse_val_um  = rmse(val_df['userPredictRating'],  predict_user_mean(val_df))
    rmse_test_um = rmse(test_df['userPredictRating'], predict_user_mean(test_df))
    mae_test_um  = mae_fn(test_df['userPredictRating'], predict_user_mean(test_df))

    print(f'User Bias: Val RMSE={rmse_val_um:.4f}, Test RMSE={rmse_test_um:.4f}, Test MAE={mae_test_um:.4f}')
    results.append({'Model': 'User Bias', 'Val RMSE': rmse_val_um,
                    'Test RMSE': rmse_test_um, 'Test MAE': mae_test_um})

In [ ]:
# Cell 4.2.3 – Level 2: Additive Bias (μ + b_u + b_i)
if DATA_AVAILABLE:
    item_mean = train_df.groupby('movieId')['userPredictRating'].mean()
    b_u = user_mean - mu
    b_i = item_mean - mu

    def predict_additive_bias(df):
        bu_vals = df['userId'].map(b_u).fillna(0).values
        bi_vals = df['movieId'].map(b_i).fillna(0).values
        return np.clip(mu + bu_vals + bi_vals, 0.5, 5.0)

    rmse_val_ab  = rmse(val_df['userPredictRating'],  predict_additive_bias(val_df))
    rmse_test_ab = rmse(test_df['userPredictRating'], predict_additive_bias(test_df))
    mae_test_ab  = mae_fn(test_df['userPredictRating'], predict_additive_bias(test_df))

    print(f'Additive Bias: Val RMSE={rmse_val_ab:.4f}, Test RMSE={rmse_test_ab:.4f}, Test MAE={mae_test_ab:.4f}')
    results.append({'Model': 'Additive Bias', 'Val RMSE': rmse_val_ab,
                    'Test RMSE': rmse_test_ab, 'Test MAE': mae_test_ab})

In [ ]:
# Cell 4.2.4 – Level 3: BiasedMF class definition (SGD, vectorized)
class BiasedMF:
    def __init__(self, n_factors=20, n_epochs=25, lr=0.005, reg=0.02,
                 clip_min=0.5, clip_max=5.0, random_state=42, patience=5):
        self.n_factors = n_factors; self.n_epochs = n_epochs
        self.lr = lr; self.reg = reg
        self.clip_min = clip_min; self.clip_max = clip_max
        self.random_state = random_state; self.patience = patience

    def _predict_batch(self, df, user_col='userId', item_col='movieId'):
        u_idxs = df[user_col].map(self.user2idx_).fillna(-1).astype(int).values
        i_idxs = df[item_col].map(self.item2idx_).fillna(-1).astype(int).values
        preds = np.full(len(df), self.mu_)
        valid = (u_idxs >= 0) & (i_idxs >= 0)
        preds[valid] = (self.mu_
                        + self.bu_[u_idxs[valid]]
                        + self.bi_[i_idxs[valid]]
                        + np.sum(self.P_[u_idxs[valid]] * self.Q_[i_idxs[valid]], axis=1))
        only_u = (u_idxs >= 0) & (i_idxs < 0)
        preds[only_u] = self.mu_ + self.bu_[u_idxs[only_u]]
        only_i = (u_idxs < 0) & (i_idxs >= 0)
        preds[only_i] = self.mu_ + self.bi_[i_idxs[only_i]]
        return np.clip(preds, self.clip_min, self.clip_max)

    def fit(self, train_df, val_df=None,
            user_col='userId', item_col='movieId', rating_col='userPredictRating'):
        rng = np.random.RandomState(self.random_state)
        users = train_df[user_col].unique()
        items = train_df[item_col].unique()
        self.user2idx_ = {u: i for i, u in enumerate(users)}
        self.item2idx_ = {v: i for i, v in enumerate(items)}
        n_u, n_i = len(users), len(items)
        self.mu_ = train_df[rating_col].mean()
        self.bu_ = np.zeros(n_u)
        self.bi_ = np.zeros(n_i)
        self.P_  = rng.normal(0, 0.01, (n_u, self.n_factors))
        self.Q_  = rng.normal(0, 0.01, (n_i, self.n_factors))
        u_arr = train_df[user_col].map(self.user2idx_).values
        i_arr = train_df[item_col].map(self.item2idx_).values
        r_arr = train_df[rating_col].values
        self.train_curve_ = []; self.val_curve_ = []
        best_val = np.inf; patience_count = 0; best_epoch = 0
        for epoch in range(self.n_epochs):
            idx    = rng.permutation(len(u_arr))
            u_s, i_s, r_s = u_arr[idx], i_arr[idx], r_arr[idx]
            preds_e = (self.mu_ + self.bu_[u_s] + self.bi_[i_s]
                       + np.sum(self.P_[u_s] * self.Q_[i_s], axis=1))
            e = r_s - preds_e
            self.bu_[u_s] += self.lr * (e - self.reg * self.bu_[u_s])
            self.bi_[i_s] += self.lr * (e - self.reg * self.bi_[i_s])
            pu = self.P_[u_s].copy(); qi = self.Q_[i_s].copy()
            self.P_[u_s] += self.lr * (e[:, None] * qi - self.reg * pu)
            self.Q_[i_s] += self.lr * (e[:, None] * pu - self.reg * qi)
            self.train_curve_.append(np.sqrt(np.mean(e**2)))
            if val_df is not None:
                vp = self._predict_batch(val_df, user_col, item_col)
                vr = np.sqrt(mean_squared_error(val_df[rating_col], vp))
                self.val_curve_.append(vr)
                if vr < best_val - 1e-4:
                    best_val = vr; best_epoch = epoch; patience_count = 0
                    self._best = (self.bu_.copy(), self.bi_.copy(),
                                  self.P_.copy(), self.Q_.copy())
                else:
                    patience_count += 1
                    if patience_count >= self.patience:
                        print(f'  Early stopping epoch {epoch+1} (best={best_epoch+1}, val={best_val:.4f})')
                        self.bu_, self.bi_, self.P_, self.Q_ = self._best
                        break
        return self

    def predict(self, df, user_col='userId', item_col='movieId'):
        return self._predict_batch(df, user_col, item_col)

print('BiasedMF class defined ✓')

In [ ]:
# Cell 4.2.5 – Hyperparameter search for BiasedMF
if DATA_AVAILABLE:
    param_grid = [
        {'n_factors': 10, 'lr': 0.01,  'reg': 0.01},
        {'n_factors': 20, 'lr': 0.005, 'reg': 0.02},
        {'n_factors': 30, 'lr': 0.005, 'reg': 0.05},
        {'n_factors': 20, 'lr': 0.01,  'reg': 0.05},
    ]
    best_config = None; best_val_rmse = np.inf
    for p in param_grid:
        m = BiasedMF(**p, n_epochs=20, patience=4, random_state=42)
        m.fit(train_df, val_df)
        vr = rmse(val_df['userPredictRating'], m.predict(val_df))
        print(f'  factors={p["n_factors"]:2d}, lr={p["lr"]}, reg={p["reg"]:5.3f} → Val RMSE={vr:.4f}')
        if vr < best_val_rmse:
            best_val_rmse = vr; best_config = p
    print(f'\nBest config: {best_config} (Val RMSE={best_val_rmse:.4f})')

In [ ]:
# Cell 4.2.6 – Train final BiasedMF + learning curve
if DATA_AVAILABLE:
    mf_model = BiasedMF(**best_config, n_epochs=40, patience=7, random_state=42)
    mf_model.fit(train_df, val_df)

    rmse_val_mf  = rmse(val_df['userPredictRating'],  mf_model.predict(val_df))
    rmse_test_mf = rmse(test_df['userPredictRating'], mf_model.predict(test_df))
    mae_test_mf  = mae_fn(test_df['userPredictRating'], mf_model.predict(test_df))
    print(f'BiasedMF: Val RMSE={rmse_val_mf:.4f}, Test RMSE={rmse_test_mf:.4f}, Test MAE={mae_test_mf:.4f}')
    results.append({'Model': 'BiasedMF', 'Val RMSE': rmse_val_mf,
                    'Test RMSE': rmse_test_mf, 'Test MAE': mae_test_mf})

    # Learning curve
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(mf_model.train_curve_, label='Train RMSE', color='#3498db')
    if mf_model.val_curve_:
        ax.plot(mf_model.val_curve_, label='Val RMSE', color='#e74c3c', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.set_title('BiasedMF Learning Curve')
    ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.2.7 – SVD++ class definition
class SVDpp:
    def __init__(self, n_factors=20, n_epochs=30, lr=0.005, reg=0.02,
                 clip_min=0.5, clip_max=5.0, random_state=42, patience=5):
        self.n_factors = n_factors; self.n_epochs = n_epochs
        self.lr = lr; self.reg = reg
        self.clip_min = clip_min; self.clip_max = clip_max
        self.random_state = random_state; self.patience = patience

    def _impl_term(self, u_idx):
        ii = self.user_items_.get(u_idx, [])
        if not ii: return np.zeros(self.n_factors)
        return np.sum(self.Y_[ii], axis=0) / np.sqrt(len(ii))

    def _predict_batch(self, df, user_col='userId', item_col='movieId'):
        u_idxs = df[user_col].map(self.user2idx_).fillna(-1).astype(int).values
        i_idxs = df[item_col].map(self.item2idx_).fillna(-1).astype(int).values
        preds  = np.full(len(df), self.mu_)
        for k in range(len(df)):
            u, i = u_idxs[k], i_idxs[k]
            if u >= 0 and i >= 0:
                impl = self._impl_term(u)
                preds[k] = (self.mu_ + self.bu_[u] + self.bi_[i]
                            + np.dot(self.P_[u] + impl, self.Q_[i]))
            elif u >= 0: preds[k] = self.mu_ + self.bu_[u]
            elif i >= 0: preds[k] = self.mu_ + self.bi_[i]
        return np.clip(preds, self.clip_min, self.clip_max)

    def fit(self, train_df, val_df=None, ratings_history_df=None,
            user_col='userId', item_col='movieId', rating_col='userPredictRating'):
        rng = np.random.RandomState(self.random_state)
        users = train_df[user_col].unique()
        items = train_df[item_col].unique()
        self.user2idx_ = {u: i for i, u in enumerate(users)}
        self.item2idx_ = {v: i for i, v in enumerate(items)}
        n_u, n_i = len(users), len(items)
        self.mu_ = train_df[rating_col].mean()
        self.bu_ = np.zeros(n_u); self.bi_ = np.zeros(n_i)
        self.P_  = rng.normal(0, 0.01, (n_u, self.n_factors))
        self.Q_  = rng.normal(0, 0.01, (n_i, self.n_factors))
        self.Y_  = rng.normal(0, 0.01, (n_i, self.n_factors))
        # Build implicit N(u) from rating history
        self.user_items_ = {}
        if ratings_history_df is not None:
            for uid, grp in ratings_history_df.groupby(user_col):
                uidx = self.user2idx_.get(uid, -1)
                if uidx < 0: continue
                ii = [self.item2idx_[m] for m in grp[item_col] if m in self.item2idx_]
                if ii: self.user_items_[uidx] = ii
        u_arr = train_df[user_col].map(self.user2idx_).values
        i_arr = train_df[item_col].map(self.item2idx_).values
        r_arr = train_df[rating_col].values
        self.train_curve_ = []; self.val_curve_ = []
        best_val = np.inf; patience_count = 0
        for epoch in range(self.n_epochs):
            idx = rng.permutation(len(u_arr)); errs = []
            for k in idx:
                u, i, r = u_arr[k], i_arr[k], r_arr[k]
                impl = self._impl_term(u)
                pu_impl = self.P_[u] + impl
                pred = self.mu_ + self.bu_[u] + self.bi_[i] + np.dot(pu_impl, self.Q_[i])
                e = r - pred; errs.append(e**2)
                self.bu_[u] += self.lr*(e - self.reg*self.bu_[u])
                self.bi_[i] += self.lr*(e - self.reg*self.bi_[i])
                qi_old = self.Q_[i].copy()
                self.P_[u] += self.lr*(e*qi_old - self.reg*self.P_[u])
                self.Q_[i] += self.lr*(e*pu_impl - self.reg*self.Q_[i])
                ii = self.user_items_.get(u, [])
                if ii:
                    norm = 1.0/np.sqrt(len(ii))
                    self.Y_[ii] += self.lr*(e*norm*qi_old - self.reg*self.Y_[ii])
            self.train_curve_.append(np.sqrt(np.mean(errs)))
            if val_df is not None:
                vp = self._predict_batch(val_df, user_col, item_col)
                vr = np.sqrt(mean_squared_error(val_df[rating_col], vp))
                self.val_curve_.append(vr)
                if vr < best_val - 1e-4:
                    best_val = vr; patience_count = 0
                    self._best = (self.bu_.copy(), self.bi_.copy(),
                                  self.P_.copy(), self.Q_.copy(), self.Y_.copy())
                else:
                    patience_count += 1
                    if patience_count >= self.patience:
                        print(f'  Early stopping epoch {epoch+1}, best val={best_val:.4f}')
                        self.bu_, self.bi_, self.P_, self.Q_, self.Y_ = self._best
                        break
        return self

    def predict(self, df, user_col='userId', item_col='movieId'):
        return self._predict_batch(df, user_col, item_col)

print('SVD++ class defined ✓')

In [ ]:
# Cell 4.2.8 – Train SVD++ + learning curve
if DATA_AVAILABLE:
    print('Training SVD++ (có thể mất vài phút)...')
    svdpp = SVDpp(n_factors=20, n_epochs=25, lr=0.005, reg=0.02, patience=5)
    svdpp.fit(train_df, val_df, ratings_history_df=ratings_raw)

    rmse_val_svd  = rmse(val_df['userPredictRating'],  svdpp.predict(val_df))
    rmse_test_svd = rmse(test_df['userPredictRating'], svdpp.predict(test_df))
    mae_test_svd  = mae_fn(test_df['userPredictRating'], svdpp.predict(test_df))
    print(f'SVD++: Val RMSE={rmse_val_svd:.4f}, Test RMSE={rmse_test_svd:.4f}, Test MAE={mae_test_svd:.4f}')
    results.append({'Model': 'SVD++', 'Val RMSE': rmse_val_svd,
                    'Test RMSE': rmse_test_svd, 'Test MAE': mae_test_svd})

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(svdpp.train_curve_, label='Train RMSE', color='#3498db')
    if svdpp.val_curve_:
        ax.plot(svdpp.val_curve_, label='Val RMSE', color='#e74c3c', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.set_title('SVD++ Learning Curve')
    ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.2.9 – RMSE Decomposition summary
if DATA_AVAILABLE:
    rq2_results = pd.DataFrame([
        {'Level': 'L0', 'Component': 'μ (Global Mean)',        'Test RMSE': rmse_test_global},
        {'Level': 'L1', 'Component': 'μ + b_u (User Bias)',    'Test RMSE': rmse_test_um},
        {'Level': 'L2', 'Component': 'μ + b_u + b_i (Additive)','Test RMSE': rmse_test_ab},
        {'Level': 'L3', 'Component': 'Biased MF (SGD)',         'Test RMSE': rmse_test_mf},
        {'Level': 'L4', 'Component': 'SVD++',                   'Test RMSE': rmse_test_svd},
    ])
    rq2_results['Δ vs L0'] = (rq2_results['Test RMSE'] - rmse_test_global).round(4)
    rq2_results['% Improve'] = ((rmse_test_global - rq2_results['Test RMSE']) / rmse_test_global * 100).round(2)
    print(rq2_results.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#95a5a6','#3498db','#2ecc71','#e67e22','#9b59b6']
    bars = ax.bar(rq2_results['Component'], rq2_results['Test RMSE'], color=colors)
    for bar, v in zip(bars, rq2_results['Test RMSE']):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.4f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Test RMSE'); ax.set_title('RQ2: Phân tách nguồn gốc Belief – Test RMSE từng Level')
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

### Nhận xét RQ2

- **L0 → L1**: Thêm user bias giảm RMSE đáng kể → xu hướng cá nhân là yếu tố chính.
- **L1 → L2**: Thêm item bias tiếp tục cải thiện → danh tiếng phim cũng ảnh hưởng.
- **L2 → L3**: BiasedMF (latent factors) cải thiện thêm → có tương tác phức tạp user-item.
- **L3 → L4**: SVD++ với implicit feedback: nếu cải thiện → lịch sử xem phim ảnh hưởng đến belief.

**Kết luận RQ2**: Belief rating được hình thành chủ yếu bởi user bias (xu hướng cá nhân), tiếp theo là item bias, và latent factors giải thích phần còn lại.

## 4.3 RQ3: Belief như Feature để Dự đoán Actual Rating

In [ ]:
# Cell 4.3.1 – Tìm matched pairs (belief trước + actual rating sau)
if DATA_AVAILABLE:
    belief_pairs = belief_raw[(belief_raw['isSeen'] == 0) &
                               belief_raw['userPredictRating'].notna()].copy()
    actual_pairs = ratings_raw[['userId','movieId','rating','timestamp']].copy()

    matched = belief_pairs.merge(actual_pairs, on=['userId','movieId'])
    matched = matched[matched['tstamp'] < matched['timestamp']]
    matched['belief_gap'] = matched['rating'] - matched['userPredictRating']

    print(f'Matched pairs (belief trước → actual sau): {len(matched):,}')
    print(f'Unique users:  {matched["userId"].nunique():,}')
    print(f'Unique movies: {matched["movieId"].nunique():,}')
    if len(matched) == 0:
        print('CẢNH BÁO: Không tìm thấy matched pairs. Kiểm tra cột tstamp/timestamp.')

In [ ]:
# Cell 4.3.2 – Phân phối belief_gap và scatter plot
if DATA_AVAILABLE and len(matched) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(matched['belief_gap'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[0].axvline(matched['belief_gap'].mean(), color='orange',
                    linestyle='--', linewidth=1.5, label=f'Mean={matched["belief_gap"].mean():.2f}')
    axes[0].set_title('Phân phối Belief Gap\n(actual − userPredictRating)')
    axes[0].set_xlabel('belief_gap'); axes[0].legend()

    sample = matched.sample(min(2000, len(matched)), random_state=42)
    axes[1].scatter(sample['userPredictRating'], sample['rating'],
                    alpha=0.3, s=15, color='#9b59b6')
    x_line = np.linspace(0.5, 5.0, 100)
    axes[1].plot(x_line, x_line, 'r--', linewidth=1.5, label='Perfect prediction')
    axes[1].set_xlabel('userPredictRating (Belief)')
    axes[1].set_ylabel('Actual Rating')
    axes[1].set_title('Belief vs Actual Rating')
    axes[1].legend()

    plt.tight_layout(); plt.show()
    corr_ba = matched[['userPredictRating','rating']].corr().iloc[0,1]
    print(f'Correlation(belief, actual) = {corr_ba:.4f}')

In [ ]:
# Cell 4.3.3 – Feature preparation for RQ3
if DATA_AVAILABLE and len(matched) > 0:
    # Merge movie stats into matched
    matched = matched.merge(movie_stats, on='movieId', how='left')
    matched = matched.merge(user_stats,  on='userId',  how='left')
    matched = matched.merge(
        movies_exp[['movieId'] + [c for c in movies_exp.columns if c.startswith('genre_')]],
        on='movieId', how='left'
    )
    g_feat_cols = [c for c in matched.columns if c.startswith('genre_')]

    BASE_FEATURES   = ['user_mean_pred','user_std_pred','user_count_pred',
                        'movie_mean_rating','movie_count_rating','movie_std_rating'] + g_feat_cols
    BELIEF_FEATURES = BASE_FEATURES + ['userPredictRating','userCertainty']

    matched[BASE_FEATURES]   = matched[BASE_FEATURES].fillna(0)
    matched[BELIEF_FEATURES] = matched[BELIEF_FEATURES].fillna(0)

    print(f'Base features:   {len(BASE_FEATURES)}')
    print(f'Belief features: {len(BELIEF_FEATURES)}')

In [ ]:
# Cell 4.3.4 – Time-based split for matched pairs
if DATA_AVAILABLE and len(matched) > 0:
    matched_sorted = matched.sort_values('tstamp')
    n = len(matched_sorted)
    n_train = int(n * 0.70); n_val = int(n * 0.15)
    m_train = matched_sorted.iloc[:n_train]
    m_val   = matched_sorted.iloc[n_train:n_train+n_val]
    m_test  = matched_sorted.iloc[n_train+n_val:]
    print(f'RQ3 split – Train: {len(m_train):,}, Val: {len(m_val):,}, Test: {len(m_test):,}')

In [ ]:
# Cell 4.3.5 – Baseline: Naive (predict belief as actual)
if DATA_AVAILABLE and len(matched) > 0 and len(m_test) > 0:
    rq3_results = {}

    naive_pred = m_test['userPredictRating'].values
    rq3_results['Naive (belief=actual)'] = {
        'RMSE': rmse(m_test['rating'], naive_pred),
        'MAE':  mae_fn(m_test['rating'], naive_pred)
    }
    print('Naive (belief as prediction):')
    print(f'  RMSE={rq3_results["Naive (belief=actual)"]["RMSE"]:.4f}')
    print(f'  MAE ={rq3_results["Naive (belief=actual)"]["MAE"]:.4f}')

In [ ]:
# Cell 4.3.6 – Ridge Regression with and without belief
if DATA_AVAILABLE and len(matched) > 0 and len(m_test) > 0:
    scaler_base   = StandardScaler().fit(m_train[BASE_FEATURES].fillna(0))
    scaler_belief = StandardScaler().fit(m_train[BELIEF_FEATURES].fillna(0))

    X_tr_b  = scaler_base.transform(m_train[BASE_FEATURES].fillna(0))
    X_val_b = scaler_base.transform(m_val[BASE_FEATURES].fillna(0))
    X_te_b  = scaler_base.transform(m_test[BASE_FEATURES].fillna(0))

    X_tr_bf  = scaler_belief.transform(m_train[BELIEF_FEATURES].fillna(0))
    X_val_bf = scaler_belief.transform(m_val[BELIEF_FEATURES].fillna(0))
    X_te_bf  = scaler_belief.transform(m_test[BELIEF_FEATURES].fillna(0))

    y_tr = m_train['rating'].values
    y_te = m_test['rating'].values

    ridge_base   = Ridge(alpha=1.0).fit(X_tr_b,  y_tr)
    ridge_belief = Ridge(alpha=1.0).fit(X_tr_bf, y_tr)

    for name, model, X_te in [('Ridge (no belief)', ridge_base, X_te_b),
                               ('Ridge (+ belief)',  ridge_belief, X_te_bf)]:
        pred = np.clip(model.predict(X_te), 0.5, 5.0)
        rq3_results[name] = {'RMSE': rmse(y_te, pred), 'MAE': mae_fn(y_te, pred)}
        print(f'{name}: RMSE={rq3_results[name]["RMSE"]:.4f}, MAE={rq3_results[name]["MAE"]:.4f}')

In [ ]:
# Cell 4.3.7 – Random Forest with and without belief
if DATA_AVAILABLE and len(matched) > 0 and len(m_test) > 0:
    rf_base   = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf_belief = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)

    rf_base.fit(m_train[BASE_FEATURES].fillna(0),   y_tr)
    rf_belief.fit(m_train[BELIEF_FEATURES].fillna(0), y_tr)

    for name, model, feats in [('RF (no belief)', rf_base,   BASE_FEATURES),
                                ('RF (+ belief)',  rf_belief, BELIEF_FEATURES)]:
        pred = np.clip(model.predict(m_test[feats].fillna(0)), 0.5, 5.0)
        rq3_results[name] = {'RMSE': rmse(y_te, pred), 'MAE': mae_fn(y_te, pred)}
        print(f'{name}: RMSE={rq3_results[name]["RMSE"]:.4f}, MAE={rq3_results[name]["MAE"]:.4f}')

In [ ]:
# Cell 4.3.8 – RQ3 bar chart comparison
if DATA_AVAILABLE and len(matched) > 0 and len(m_test) > 0:
    rq3_df = pd.DataFrame([
        {'Model': k, 'RMSE': v['RMSE'], 'MAE': v['MAE']}
        for k, v in rq3_results.items()
    ])
    print(rq3_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    palette = ['#95a5a6','#e74c3c','#27ae60','#e67e22','#9b59b6']
    for ax, metric in zip(axes, ['RMSE','MAE']):
        bars = ax.bar(rq3_df['Model'], rq3_df[metric],
                      color=palette[:len(rq3_df)], edgecolor='white')
        for bar, v in zip(bars, rq3_df[metric]):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.4f}',
                    ha='center', va='bottom', fontsize=9)
        ax.set_title(f'RQ3: So sánh {metric} – với/không belief')
        ax.set_ylabel(metric)
        plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.3.9 – Feature importance from RF with belief
if DATA_AVAILABLE and len(matched) > 0 and len(m_test) > 0:
    importances = pd.Series(rf_belief.feature_importances_, index=BELIEF_FEATURES)
    top_feats = importances.nlargest(20).sort_values()

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = ['#e74c3c' if 'Predict' in f or 'Certainty' in f or 'belief' in f.lower()
              else '#3498db' for f in top_feats.index]
    top_feats.plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Top 20 Feature Importance – RF (+ belief)\n(đỏ = belief-related features)', fontsize=12)
    ax.set_xlabel('Feature Importance')
    plt.tight_layout(); plt.show()

### Nhận xét RQ3

- **Naive baseline**: Dùng trực tiếp belief làm dự đoán actual rating.
- **Ridge/RF không có belief**: Chỉ dùng user/item statistics và genre features.
- **Ridge/RF có belief**: Thêm `userPredictRating` và `userCertainty` vào feature set.

**Kết luận RQ3**: Nếu RMSE giảm khi thêm belief features → Belief có giá trị dự đoán thực tế. Feature importance cho thấy `userPredictRating` (belief) có vị trí như thế nào so với các features khác.

---
# 5. Đánh Giá Model

## 5.1 Mẫu dự đoán (10 ngẫu nhiên từ test set)

In [ ]:
# Cell 5.1 – Sample predictions
if DATA_AVAILABLE and len(test_df) > 0:
    sample_idx = np.random.RandomState(0).choice(len(test_df), min(10, len(test_df)), replace=False)
    sample = test_df.iloc[sample_idx].copy()

    sample['pred_global'] = mu
    sample['pred_userbias'] = predict_user_mean(sample)
    sample['pred_mf']     = mf_model.predict(sample)
    sample['pred_svdpp']  = svdpp.predict(sample)

    display_cols = ['userId','movieId','userPredictRating','pred_global',
                    'pred_userbias','pred_mf','pred_svdpp']
    display(sample[display_cols].round(3))

## 5.2 Bảng so sánh tổng thể

In [ ]:
# Cell 5.2 – Full comparison table + grouped bar chart
if DATA_AVAILABLE:
    results_df = pd.DataFrame(results)
    results_df['% Improve vs Baseline'] = (
        (results_df['Test RMSE'].iloc[0] - results_df['Test RMSE'])
        / results_df['Test RMSE'].iloc[0] * 100
    ).round(2)
    print('=== Bảng so sánh tổng thể ===')
    print(results_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(results_df))
    w = 0.35
    b1 = ax.bar(x - w/2, results_df['Val RMSE'],  w, label='Val RMSE',  color='#3498db', alpha=0.85)
    b2 = ax.bar(x + w/2, results_df['Test RMSE'], w, label='Test RMSE', color='#e74c3c', alpha=0.85)
    for bar in list(b1)+list(b2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
    ax.set_ylabel('RMSE'); ax.set_title('So sánh Val/Test RMSE giữa các mô hình')
    ax.legend()
    plt.tight_layout(); plt.show()

## 5.3 Phân tích lỗi (Error Analysis)

In [ ]:
# Cell 5.3.1 – Scatter: predicted vs actual
if DATA_AVAILABLE and len(test_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, (name, preds) in zip(axes, [
        ('Additive Bias', predict_additive_bias(test_df)),
        ('BiasedMF',      mf_model.predict(test_df))
    ]):
        ax.scatter(preds, test_df['userPredictRating'].values,
                   alpha=0.2, s=8, color='#3498db')
        mn, mx = 0.5, 5.0
        ax.plot([mn,mx],[mn,mx],'r--',linewidth=1.5,label='Perfect')
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual Belief Rating')
        ax.set_title(f'{name} – Predicted vs Actual')
        ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 5.3.2 – Error by certainty level
if DATA_AVAILABLE and len(test_df) > 0:
    test_err = test_df[['userCertainty','userPredictRating']].copy()
    test_err['err_mf']    = np.abs(mf_model.predict(test_df) - test_df['userPredictRating'].values)
    test_err['err_global']= np.abs(mu - test_df['userPredictRating'].values)

    err_by_cert = test_err.groupby('userCertainty')[['err_mf','err_global']].mean()

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(err_by_cert))
    w = 0.35
    ax.bar(x-w/2, err_by_cert['err_global'], w, label='Global Mean MAE', color='#95a5a6')
    ax.bar(x+w/2, err_by_cert['err_mf'],     w, label='BiasedMF MAE',    color='#e74c3c')
    ax.set_xticks(x); ax.set_xticklabels(err_by_cert.index)
    ax.set_xlabel('userCertainty')
    ax.set_ylabel('Mean Absolute Error')
    ax.set_title('MAE theo mức Certainty – Global Mean vs BiasedMF')
    ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 5.3.3 – Error histogram
if DATA_AVAILABLE and len(test_df) > 0:
    errors_mf = mf_model.predict(test_df) - test_df['userPredictRating'].values

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(errors_mf, bins=40, color='#3498db', edgecolor='white', alpha=0.8)
    ax.axvline(0,                    color='red',    linestyle='--', linewidth=1.5, label='Zero error')
    ax.axvline(np.mean(errors_mf),   color='orange', linestyle='--', linewidth=1.5,
               label=f'Mean={np.mean(errors_mf):.3f}')
    ax.set_title('Phân phối lỗi (BiasedMF): predicted − actual')
    ax.set_xlabel('Error')
    ax.legend()
    plt.tight_layout(); plt.show()

## 5.4 Ablation: Kiểm tra Data Leakage (systemPredictRating)

In [ ]:
# Cell 5.4 – Ablation: what if we include systemPredictRating?
if DATA_AVAILABLE and 'systemPredictRating' in belief_feat.columns:
    sys_col_available = belief_feat['systemPredictRating'].notna().sum() > 0
    if sys_col_available:
        X_sys_train = train_df[['systemPredictRating']].fillna(mu)
        X_sys_test  = test_df[['systemPredictRating']].fillna(mu)
        y_train_all = train_df['userPredictRating'].values
        y_test_all  = test_df['userPredictRating'].values

        ridge_sys = Ridge(alpha=1.0).fit(X_sys_train, y_train_all)
        pred_sys  = np.clip(ridge_sys.predict(X_sys_test), 0.5, 5.0)
        rmse_sys  = rmse(y_test_all, pred_sys)
        print(f'Ridge chỉ dùng systemPredictRating: Test RMSE = {rmse_sys:.4f}')
        print(f'(So với Global Mean: {rmse_test_global:.4f})')
        print()
        corr_sys = belief_feat[['userPredictRating','systemPredictRating']].dropna().corr().iloc[0,1]
        print(f'Correlation(userPredictRating, systemPredictRating) = {corr_sys:.4f}')
        print('Nếu correlation cao → systemPredictRating có thể chứa thông tin từ actual rating (leakage risk)')
    else:
        print('systemPredictRating không có giá trị hoặc toàn NaN')
else:
    print('systemPredictRating không có trong dữ liệu hoặc DATA_AVAILABLE=False')

---
# 6. Kết Luận Chung

## 6.1 Tóm tắt kết quả theo từng câu hỏi nghiên cứu

### RQ1 – Belief Gap theo Thể Loại
> Người dùng **không** có xu hướng đánh giá đồng đều giữa các thể loại. Các thể loại 'nghiêm túc' (Documentary, Film-Noir) có xu hướng bị under-estimate, trong khi các thể loại 'giải trí nhẹ' (Animation, Fantasy) bị over-estimate. Certainty cao tương quan với belief gap nhỏ hơn, cho thấy sự calibration tốt hơn.

### RQ2 – Phân tích nguồn gốc Belief
> Belief rating được hình thành chủ yếu bởi **user bias** (xu hướng cá nhân), tiếp theo là **item bias** (danh tiếng phim). Latent factors (BiasedMF, SVD++) cải thiện thêm RMSE, cho thấy tương tác user-item phức tạp đóng vai trò không nhỏ. SVD++ với implicit feedback từ rating history cho kết quả tốt nhất.

### RQ3 – Belief như Feature
> Thêm `userPredictRating` và `userCertainty` vào feature set **cải thiện** khả năng dự đoán actual rating (RMSE giảm), đặc biệt với Random Forest. Điều này chứng tỏ belief rating mang thông tin có giá trị về preferences của người dùng.

## 6.2 Hạn chế và Hướng phát triển

**Hạn chế**:
- Số lượng matched pairs (belief trước → actual rating sau) có thể hạn chế
- SVD++ vẫn dùng SGD sequential, chưa tối ưu cho tập dữ liệu lớn
- Chưa xem xét cold-start problem cho user/item mới

**Hướng phát triển**:
- Thử nghiệm Neural Collaborative Filtering (NCF) hoặc Transformer-based models
- Áp dụng Bayesian Personalized Ranking (BPR) cho bài toán ranking
- Phân tích temporal dynamics: belief có thay đổi theo thời gian không?
- Tích hợp thêm dữ liệu ngữ cảnh (context-aware RS)

## 6.3 Đóng góp chính

1. **Framework phân tích belief gap** theo thể loại và certainty level
2. **Decomposition pipeline** từ Global Mean → BiasedMF để đo đóng góp của từng thành phần
3. **Empirical evidence** rằng belief rating là feature hữu ích cho dự đoán actual rating

---
*Notebook này được tạo cho bài thi cuối kỳ môn Giới thiệu về Máy học – UEL University*

In [ ]:
# Cell 6.1 – Final summary printout
if DATA_AVAILABLE:
    print('='*60)
    print('FINAL RESULTS SUMMARY')
    print('='*60)
    print('\n[RQ2] Belief Rating Prediction (test set):')
    if 'results_df' in dir():
        print(results_df[['Model','Test RMSE','Test MAE']].to_string(index=False))
    if 'rq3_results' in dir() and rq3_results:
        print('\n[RQ3] Actual Rating Prediction (matched pairs, test set):')
        for model_name, metrics in rq3_results.items():
            print(f'  {model_name:<30} RMSE={metrics["RMSE"]:.4f}  MAE={metrics["MAE"]:.4f}')
    print('='*60)
    print('\nKết luận: Xem Section 6 để biết chi tiết')